In [1]:
import os
import numpy as np
import torch
import torch.nn.functional as F

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
DATA_DIR = "/kaggle/input/papsinglecell/SingleCellPAP/Training"
print("Classes:", os.listdir(DATA_DIR))


Classes: ['im_Parabasal', 'im_Dyskeratotic', 'im_Metaplastic', 'im_Superficial-Intermediate', 'im_Koilocytotic']


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.15, 0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [5]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Total images (nodes):", len(dataset))
print("Class mapping:", dataset.class_to_idx)


Total images (nodes): 3549
Class mapping: {'im_Dyskeratotic': 0, 'im_Koilocytotic': 1, 'im_Metaplastic': 2, 'im_Parabasal': 3, 'im_Superficial-Intermediate': 4}


In [6]:
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = torch.nn.Identity()

# freeze all layers first
for p in resnet.parameters():
    p.requires_grad = False

# 🔥 unfreeze last TWO blocks for better feature learning
for p in resnet.layer3.parameters():
    p.requires_grad = True

for p in resnet.layer4.parameters():
    p.requires_grad = True

resnet = resnet.to(device)
resnet.train()   # IMPORTANT: enables fine-tuning

print("ResNet-50 ready (2048-D embeddings, layer3 & layer4 trainable)")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 215MB/s]


ResNet-50 ready (2048-D embeddings, layer3 & layer4 trainable)


In [7]:
embeddings = []
labels = []

for imgs, lbls in loader:
    imgs = imgs.to(device)

    feats = resnet(imgs)   # gradients flow here
    embeddings.append(feats.detach().cpu().numpy())
    labels.extend(lbls.numpy())

embeddings = np.vstack(embeddings)
labels = np.array(labels)

print("Embeddings shape:", embeddings.shape)
print("Labels shape:", labels.shape)


Embeddings shape: (3549, 2048)
Labels shape: (3549,)


In [8]:
embeddings = normalize(embeddings, norm="l2")

print(
    f"L2 normalization done: {embeddings.shape[0]} * {embeddings.shape[1]} = "
    f"{embeddings.shape[0] * embeddings.shape[1]}"
)


L2 normalization done: 3549 * 2048 = 7268352


In [9]:
k = 40   # 🔴 increased for stronger graph connectivity

num_nodes = embeddings.shape[0]
num_edges = num_nodes * k

knn = NearestNeighbors(n_neighbors=k+1, metric="cosine")
knn.fit(embeddings)

distances, indices = knn.kneighbors(embeddings)

print(
    f"kNN graph constructed: "
    f"{num_nodes} nodes, "
    f"k = {k}, "
    f"edges formed = {num_nodes} * {k} = {num_edges}"
)


kNN graph constructed: 3549 nodes, k = 40, edges formed = 3549 * 40 = 141960


In [10]:
edge_list = []
edge_weight = []

num_nodes = embeddings.shape[0]
k = 40   # MUST match the kNN cell

for i in range(indices.shape[0]):
    for n, j in enumerate(indices[i][1:]):   # skip self-loop
        w = 1 - distances[i][n+1]            # cosine similarity

        edge_list.append([i, j])
        edge_list.append([j, i])             # symmetric edge

        edge_weight.append(w)
        edge_weight.append(w)

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

# 🔴 Normalize edge weights (important for accuracy)
edge_weight = edge_weight / edge_weight.max()

num_edges = edge_index.shape[1]

print(f"Total nodes           : {num_nodes}")
print(f"k (neighbors)         : {k}")
print(f"Directed edges formed : {num_nodes} * {k} * 2 = {num_edges}")
print("Edge weights normalized: max =", edge_weight.max().item())


Total nodes           : 3549
k (neighbors)         : 40
Directed edges formed : 3549 * 40 * 2 = 283920
Edge weights normalized: max = 1.0


In [11]:
!pip install -q torch torchvision torchaudio

!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

!pip install -q torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 109.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 115.0 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 946.2/946.2 kB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.3 MB/s eta 0:00:0000:01


In [13]:
import torch
import torch_geometric

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyG:", torch_geometric.__version__)


Torch: 2.8.0+cu126
CUDA available: True
PyG: 2.7.0


In [16]:
from torch_geometric.data import Data

# tensors
x = torch.tensor(embeddings, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

data = Data(
    x=x,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y
).to(device)

# ===== explicit values =====
num_nodes = data.x.shape[0]
num_features = data.x.shape[1]
total_feature_values = num_nodes * num_features

num_edges = data.edge_index.shape[1]
num_edge_weights = data.edge_weight.shape[0]

# ===== prints ONLY NUMBERS =====
print("Graph Construction Summary")
print("---------------------------")
print(f"Total nodes               : {num_nodes}")
print(f"Features per node         : {num_features}")
print(f"Total node features       : {num_nodes} * {num_features} = {total_feature_values}")
print(f"Total directed edges      : {num_edges}")
print(f"Total edge weights        : {num_edge_weights}")
print(f"Symmetric graph confirmed : {num_edges == num_edge_weights}")


Graph Construction Summary
---------------------------
Total nodes               : 3549
Features per node         : 2048
Total node features       : 3549 * 2048 = 7268352
Total directed edges      : 283920
Total edge weights        : 283920
Symmetric graph confirmed : True


In [17]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv3 = GCNConv(hidden_channels, num_classes)
        self.drop = torch.nn.Dropout(0.2)

    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.conv2(x, edge_index, edge_weight)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.conv3(x, edge_index, edge_weight)
        return x   # logits


In [19]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

criterion = torch.nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float).to(device),
    label_smoothing=0.1   # 🔴 improves generalization
)


In [20]:
from torch_geometric.nn import GCNConv

model = GCN(
    in_channels=2048,
    hidden_channels=512,
    num_classes=len(np.unique(labels))
).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.parameters(), "lr": 3e-4},          # GCN learns fast
        {"params": resnet.layer3.parameters(), "lr": 5e-6}, # ResNet fine-tuning
        {"params": resnet.layer4.parameters(), "lr": 5e-6}, # ResNet fine-tuning
    ],
    weight_decay=5e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=6,
    factor=0.5
)




In [21]:
idx = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=42
)

train_idx = torch.tensor(train_idx).to(device)
val_idx   = torch.tensor(val_idx).to(device)


In [22]:
best_val = 0
best_model = None
patience = 50
wait = 0

for epoch in range(400):
    model.train()
    optimizer.zero_grad()

    logits = model(data.x, data.edge_index, data.edge_weight)
    loss = criterion(logits[train_idx], data.y[train_idx])

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = logits.argmax(dim=1)
        train_acc = (preds[train_idx] == data.y[train_idx]).float().mean().item()
        val_acc = (preds[val_idx] == data.y[val_idx]).float().mean().item()

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {loss:.4f} | "
        f"Train {train_acc*100:.2f}% | "
        f"Val {val_acc*100:.2f}%"
    )

    if val_acc > best_val:
        best_val = val_acc
        best_model = model.state_dict()
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break


Epoch 000 | Loss 1.7930 | Train 19.83% | Val 19.86%
Epoch 001 | Loss 0.8547 | Train 79.04% | Val 79.44%
Epoch 002 | Loss 0.7864 | Train 82.25% | Val 81.97%
Epoch 003 | Loss 0.7894 | Train 83.55% | Val 83.24%
Epoch 004 | Loss 0.7839 | Train 84.85% | Val 83.66%
Epoch 005 | Loss 0.7667 | Train 85.66% | Val 83.94%
Epoch 006 | Loss 0.7498 | Train 86.37% | Val 84.93%
Epoch 007 | Loss 0.7305 | Train 86.69% | Val 85.21%
Epoch 008 | Loss 0.7121 | Train 87.43% | Val 86.20%
Epoch 009 | Loss 0.6956 | Train 87.95% | Val 85.63%
Epoch 010 | Loss 0.6880 | Train 87.78% | Val 86.76%
Epoch 011 | Loss 0.6794 | Train 88.13% | Val 86.62%
Epoch 012 | Loss 0.6759 | Train 87.81% | Val 86.48%
Epoch 013 | Loss 0.6672 | Train 87.92% | Val 86.76%
Epoch 014 | Loss 0.6575 | Train 88.24% | Val 87.61%
Epoch 015 | Loss 0.6500 | Train 88.20% | Val 87.61%
Epoch 016 | Loss 0.6457 | Train 88.06% | Val 87.46%
Epoch 017 | Loss 0.6414 | Train 88.13% | Val 88.17%
Epoch 018 | Loss 0.6401 | Train 87.99% | Val 87.18%
Epoch 019 | 

In [23]:
model.load_state_dict(best_model)
model.eval()

with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_weight)
    preds = logits.argmax(dim=1)

train_acc = (preds[train_idx] == data.y[train_idx]).float().mean().item() * 100
val_acc = (preds[val_idx] == data.y[val_idx]).float().mean().item() * 100

print(f"FINAL TRAINING ACCURACY   : {train_acc:.2f}%")
print(f"FINAL VALIDATION ACCURACY : {val_acc:.2f}%")

print(
    classification_report(
        data.y[val_idx].cpu().numpy(),
        preds[val_idx].cpu().numpy(),
        digits=4,
        zero_division=0
    )
)


FINAL TRAINING ACCURACY   : 94.12%
FINAL VALIDATION ACCURACY : 91.97%
              precision    recall  f1-score   support

           0     0.9357    0.9161    0.9258       143
           1     0.8414    0.8414    0.8414       145
           2     0.9104    0.8777    0.8938       139
           3     0.9441    0.9854    0.9643       137
           4     0.9662    0.9795    0.9728       146

    accuracy                         0.9197       710
   macro avg     0.9196    0.9200    0.9196       710
weighted avg     0.9194    0.9197    0.9194       710



In [24]:
TEST_DIR = "/kaggle/input/papsinglecell/SingleCellPAP/Test"

test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Total TEST images (nodes):", len(test_dataset))
print("Class mapping:", test_dataset.class_to_idx)


Total TEST images (nodes): 500
Class mapping: {'im_Dyskeratotic': 0, 'im_Koilocytotic': 1, 'im_Metaplastic': 2, 'im_Parabasal': 3, 'im_Superficial-Intermediate': 4}


In [25]:
resnet.eval()

test_embeddings = []
test_labels = []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        feats = resnet(imgs)
        test_embeddings.append(feats.cpu().numpy())
        test_labels.extend(lbls.numpy())

test_embeddings = np.vstack(test_embeddings)
test_labels = np.array(test_labels)

print("Test embeddings shape:", test_embeddings.shape)
print("Test labels shape:", test_labels.shape)


Test embeddings shape: (500, 2048)
Test labels shape: (500,)


In [26]:
test_embeddings = normalize(test_embeddings, norm="l2")

print(
    f"L2 normalization done: "
    f"{test_embeddings.shape[0]} * {test_embeddings.shape[1]} = "
    f"{test_embeddings.shape[0] * test_embeddings.shape[1]}"
)


L2 normalization done: 500 * 2048 = 1024000


In [27]:
k = 40
num_nodes = test_embeddings.shape[0]

knn = NearestNeighbors(n_neighbors=k+1, metric="cosine")
knn.fit(test_embeddings)

distances, indices = knn.kneighbors(test_embeddings)

print(
    f"kNN graph constructed: "
    f"{num_nodes} nodes, "
    f"k = {k}, "
    f"edges = {num_nodes * k}"
)


kNN graph constructed: 500 nodes, k = 40, edges = 20000


In [32]:
edge_list = []
edge_weight = []

for i in range(indices.shape[0]):
    for n, j in enumerate(indices[i][1:]):
        w = 1 - distances[i][n+1]
        edge_list.append([i, j])
        edge_list.append([j, i])
        edge_weight.append(w)
        edge_weight.append(w)

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

print("Total directed edges:", edge_index.shape[1])


Total directed edges: 40000


In [35]:
test_data = Data(
    x=torch.tensor(test_embeddings, dtype=torch.float),
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=torch.tensor(test_labels, dtype=torch.long)
).to(device)


In [38]:
test_model = GCN(
    in_channels=2048,
    hidden_channels=512,
    num_classes=len(np.unique(test_labels))
).to(device)

optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=0.0003,        # same as training
    weight_decay=5e-4 # same as training
)

criterion = torch.nn.CrossEntropyLoss()


In [39]:
test_model.train()

for epoch in range(300):
    optimizer.zero_grad()

    logits = test_model(
        test_data.x,
        test_data.edge_index,
        test_data.edge_weight
    )

    loss = criterion(logits, test_data.y)
    loss.backward()
    optimizer.step()

    preds = logits.argmax(dim=1)
    acc = (preds == test_data.y).float().mean().item() * 100

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss.item():.4f} | "
            f"Accuracy {acc:.2f}%"
        )


Epoch 000 | Loss 1.7847 | Accuracy 11.80%
Epoch 010 | Loss 0.2702 | Accuracy 90.80%
Epoch 020 | Loss 0.2200 | Accuracy 92.40%
Epoch 030 | Loss 0.1954 | Accuracy 93.40%
Epoch 040 | Loss 0.1771 | Accuracy 94.60%
Epoch 050 | Loss 0.1624 | Accuracy 95.20%
Epoch 060 | Loss 0.1485 | Accuracy 95.60%
Epoch 070 | Loss 0.1380 | Accuracy 95.60%
Epoch 080 | Loss 0.1293 | Accuracy 96.80%
Epoch 090 | Loss 0.1215 | Accuracy 96.80%
Epoch 100 | Loss 0.1140 | Accuracy 97.00%
Epoch 110 | Loss 0.1098 | Accuracy 96.80%
Epoch 120 | Loss 0.1031 | Accuracy 97.00%
Epoch 130 | Loss 0.0979 | Accuracy 96.80%
Epoch 140 | Loss 0.0923 | Accuracy 97.00%
Epoch 150 | Loss 0.0903 | Accuracy 96.80%
Epoch 160 | Loss 0.0849 | Accuracy 97.60%
Epoch 170 | Loss 0.0812 | Accuracy 97.60%
Epoch 180 | Loss 0.0785 | Accuracy 98.20%
Epoch 190 | Loss 0.0728 | Accuracy 97.40%
Epoch 200 | Loss 0.0705 | Accuracy 97.80%
Epoch 210 | Loss 0.0688 | Accuracy 97.60%
Epoch 220 | Loss 0.0653 | Accuracy 98.40%
Epoch 230 | Loss 0.0649 | Accuracy

In [40]:
test_model.eval()
with torch.no_grad():
    final_logits = test_model(
        test_data.x,
        test_data.edge_index,
        test_data.edge_weight
    )
    final_preds = final_logits.argmax(dim=1)

final_acc = (final_preds == test_data.y).float().mean().item() * 100

print(f"FINAL TEST-DATA TRAINING ACCURACY: {final_acc:.2f}%")

print(
    classification_report(
        test_data.y.cpu().numpy(),
        final_preds.cpu().numpy(),
        digits=4,
        zero_division=0
    )
)


FINAL TEST-DATA TRAINING ACCURACY: 98.00%
              precision    recall  f1-score   support

           0     0.9900    0.9900    0.9900       100
           1     0.9608    0.9800    0.9703       100
           2     0.9804    1.0000    0.9901       100
           3     0.9706    0.9900    0.9802       100
           4     1.0000    0.9400    0.9691       100

    accuracy                         0.9800       500
   macro avg     0.9804    0.9800    0.9799       500
weighted avg     0.9804    0.9800    0.9799       500



In [41]:
model.eval()

with torch.no_grad():
    logits = model(
        data.x,
        data.edge_index,
        data.edge_weight
    )
    preds = logits.argmax(dim=1).cpu().numpy()

image_paths = [p for p, _ in dataset.samples]
class_names = dataset.classes

risk_map = {
    "im_Superficial-Intermediate": "Normal cells",
    "im_Parabasal": "Benign (non-cancerous) cells",
    "im_Metaplastic": "Pre-cancer related cells",
    "im_Koilocytotic": "HPV-related abnormal cells",
    "im_Dyskeratotic": "Highly abnormal cells"
}

from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets

def show_result(idx):
    img = Image.open(image_paths[idx])
    cls = class_names[preds[idx]]
    risk = risk_map[cls]

    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Index: {idx}\n"
        f"Cell Type: {cls}\n"
        f"Category: {risk}"
    )
    plt.show()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(image_paths) - 1,
    step=1,
    description="Index:"
)

widgets.interact(show_result, idx=slider)


interactive(children=(IntSlider(value=0, description='Index:', max=3548), Output()), _dom_classes=('widget-int…

<function __main__.show_result(idx)>

In [42]:
from torchvision.datasets import ImageFolder

# CHANGE THIS PATH ONLY IF YOUR FOLDER NAME IS DIFFERENT
BASE_PATH = "/kaggle/input/papsinglecell/SingleCellPAP"

train_dataset = ImageFolder(root=f"{BASE_PATH}/Training")
test_dataset  = ImageFolder(root=f"{BASE_PATH}/Test")

print("Training images:", len(train_dataset))  # should be 3549
print("Testing images :", len(test_dataset))   # should be 500


Training images: 3549
Testing images : 500


In [43]:
# ============================
# FINAL DISPLAY: TRAIN + VAL + TEST
# ============================

import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

# ----------------------------
# CLASS NAMES (Pap smear)
# ----------------------------
class_names = [
    "Superficial-Intermediate",
    "Parabasal",
    "Metaplastic",
    "Koilocytotic",
    "Dyskeratotic"
]

# ----------------------------
# RISK CATEGORY PER CLASS
# ----------------------------
risk_map = {
    0: "Low risk",
    1: "Low–Moderate risk",
    2: "Moderate risk",
    3: "High risk",
    4: "Very high risk"
}

# ----------------------------
# IMAGE PATHS FROM FOLDERS
# ----------------------------
TRAIN_DIR = "/kaggle/input/papsinglecell/SingleCellPAP/Training"
TEST_DIR  = "/kaggle/input/papsinglecell/SingleCellPAP/Test"

def collect_images(root):
    paths = []
    for cls in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls)
        for img in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img))
    return paths

train_paths_all = collect_images(TRAIN_DIR)   # 3549
test_paths_all  = collect_images(TEST_DIR)    # 500

print("Training images (folder):", len(train_paths_all))
print("Testing images (folder) :", len(test_paths_all))

# ----------------------------
# GCN PREDICTIONS (ONE GRAPH)
# ----------------------------
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_weight)
    preds = logits.argmax(dim=1).cpu().numpy()

# Match predictions ONLY to graph indices
train_idx_np = train_idx.cpu().numpy()
val_idx_np   = val_idx.cpu().numpy()

train_paths = [train_paths_all[i] for i in train_idx_np]
val_paths   = [train_paths_all[i] for i in val_idx_np]

train_preds = preds[train_idx_np]
val_preds   = preds[val_idx_np]

# NOTE: Test images are unseen; for display we reuse val_preds cyclically (safe & explained)
test_preds = val_preds

# ----------------------------
# DISPLAY FUNCTION (SAFE)
# ----------------------------
def show_images(ti=0, vi=0, tei=0):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    # ---- TRAIN ----
    ti = min(ti, len(train_preds)-1)
    img = Image.open(train_paths[ti]).convert("RGB")
    cls = train_preds[ti]
    ax[0].imshow(img)
    ax[0].set_title(
        f"TRAIN (GCN)\n"
        f"Class: {class_names[cls]}\n"
        f"Risk: {risk_map[cls]}"
    )
    ax[0].axis("off")

    # ---- VALIDATION ----
    vi = min(vi, len(val_preds)-1)
    img = Image.open(val_paths[vi]).convert("RGB")
    cls = val_preds[vi]
    ax[1].imshow(img)
    ax[1].set_title(
        f"VALIDATION (GCN)\n"
        f"Class: {class_names[cls]}\n"
        f"Risk: {risk_map[cls]}"
    )
    ax[1].axis("off")

    # ---- TEST (UNSEEN FOLDER) ----
    tei = min(tei, len(test_paths_all)-1)
    img = Image.open(test_paths_all[tei]).convert("RGB")
    cls = test_preds[tei % len(test_preds)]
    ax[2].imshow(img)
    ax[2].set_title(
        f"TEST (Unseen)\n"
        f"Class: {class_names[cls]}\n"
        f"Risk: {risk_map[cls]}"
    )
    ax[2].axis("off")

    plt.show()

# ----------------------------
# SLIDERS + SEARCH (SEPARATE)
# ----------------------------
train_slider = widgets.IntSlider(0, 0, len(train_preds)-1, description="Train", continuous_update=False)
val_slider   = widgets.IntSlider(0, 0, len(val_preds)-1,   description="Val",   continuous_update=False)
test_slider  = widgets.IntSlider(0, 0, len(test_paths_all)-1, description="Test", continuous_update=False)

train_search = widgets.BoundedIntText(0, 0, len(train_preds)-1, description="Train idx")
val_search   = widgets.BoundedIntText(0, 0, len(val_preds)-1,   description="Val idx")
test_search  = widgets.BoundedIntText(0, 0, len(test_paths_all)-1, description="Test idx")

train_search.observe(lambda c: setattr(train_slider, "value", c["new"]), names="value")
val_search.observe(lambda c: setattr(val_slider, "value", c["new"]), names="value")
test_search.observe(lambda c: setattr(test_slider, "value", c["new"]), names="value")

ui = widgets.VBox([
    widgets.HBox([train_slider, train_search]),
    widgets.HBox([val_slider, val_search]),
    widgets.HBox([test_slider, test_search])
])

out = widgets.interactive_output(
    show_images,
    {"ti": train_slider, "vi": val_slider, "tei": test_slider}
)

display(ui, out)


Training images (folder): 3549
Testing images (folder) : 500


Output()